### Install packages

In [ ]:
!pip uninstall -y transformers
!pip install -U "transformers==5.5.1" datasets evaluate accelerate

Found existing installation: transformers 5.5.1
Uninstalling transformers-5.5.1:
  Successfully uninstalled transformers-5.5.1
  Using cached transformers-5.5.1-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.1-py3-none-any.whl (10.2 MB)


In [ ]:
%pip install -U datasets huggingface_hub

  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-2.2.0-py3-none-any.whl (839 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.5.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.2.0 which is incompatible.
tokenizers 0.22.2 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.2.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.2.0 which is incompatible.
gradio-client 2.7.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.2.0 which is incompatible.


In [ ]:
!pip install -U peft

  Using cached huggingface_hub-1.33.0-py3-none-any.whl.metadata (16 kB)
Using cached huggingface_hub-1.33.0-py3-none-any.whl (846 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.2.0
    Uninstalling huggingface_hub-2.2.0:
      Successfully uninstalled huggingface_hub-2.2.0


### Imports and setup

In [ ]:
import os
import json
import math
import random
import collections
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
import evaluate
from tqdm.auto import tqdm

# -----------------------------
# Reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -----------------------------
# Config
# -----------------------------
MODEL_CHECKPOINT = "bert-base-uncased"   # change to "bert-base-cased" if you want cased BERT
OUTPUT_DIR = "/content/bert_squad_finetuned"

MAX_LENGTH = 384  # Maximum input sequence length, including special tokens
DOC_STRIDE = 128  # Overlap between context windows when a paragraph is split
N_BEST = 20       # Number of high-scoring start/end candidates considered
MAX_ANSWER_LENGTH = 30  # Maximum answer span in tokens

# Set True first if you want a quick sanity-check run on Colab
USE_SMALL_SUBSET = True

# Training hyperparameters
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
NUM_EPOCHS = 1
WEIGHT_DECAY = 0.01

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


### Load SQuAD ( Standard Question and Answering)

In [ ]:
# Uses only 20,000 training examples and 1,000 validation examples
raw_datasets = load_dataset("rajpurkar/squad")

if USE_SMALL_SUBSET:
    raw_datasets["train"] = raw_datasets["train"].select(range(20000))
    raw_datasets["validation"] = raw_datasets["validation"].select(range(1000))

print(raw_datasets)
print(raw_datasets["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 1000
    })
})
{'id': '5733be284776f41900661182', 'title': 'University_of_Notre_Dame', 'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome)

### Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

### Preprocess training data

In [ ]:
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If answer is not fully inside the context window
        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise find token start
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            # And token end
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

### Preprocess validation data

In [ ]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offsets = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offsets)
        ]

    inputs["example_id"] = example_ids
    return inputs

### Tokenize datasets

In [ ]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

# This one is for Trainer evaluation loss
eval_dataset_for_loss = raw_datasets["validation"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

# This one is for final EM/F1 post-processing
validation_features = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

print(train_dataset)
print(eval_dataset_for_loss)
print(validation_features)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 20164
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 1020
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
    num_rows: 1020
})


### Model, data collator, and metric

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)
data_collator = DefaultDataCollator()
metric = evaluate.load("squad")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized beca

### LoRA (Low-Rank Adaptation)

### Post-processing for EM/F1 on SQuAD

In [ ]:
def compute_squad_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []

    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip if outside context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue

                    # Skip invalid spans
                    if end_index < start_index or (end_index - start_index + 1) > MAX_ANSWER_LENGTH:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    answers.append({
                        "text": context[start_char:end_char],
                        "logit_score": float(start_logit[start_index] + end_logit[end_index]),
                    })

        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append({
                "id": example_id,
                "prediction_text": best_answer["text"]
            })
        else:
            predicted_answers.append({
                "id": example_id,
                "prediction_text": ""
            })

    theoretical_answers = [
        {"id": ex["id"], "answers": ex["answers"]} for ex in examples
    ]

    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

### Training arguments and trainer

In [ ]:
import os
import shutil
import torch
from transformers import TrainingArguments

# Delete old output folder manually, since overwrite_output_dir is not used here
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=2,
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_loss,
    data_collator=data_collator,
    processing_class=tokenizer,
)

### Train

In [ ]:
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete.")
print(train_result)

Epoch,Training Loss,Validation Loss
1,1.255016,1.239552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete.
TrainOutput(global_step=5041, training_loss=1.499718804369841, metrics={'train_runtime': 534.1651, 'train_samples_per_second': 37.749, 'train_steps_per_second': 9.437, 'total_flos': 3951590852118528.0, 'train_loss': 1.499718804369841, 'epoch': 1.0})


In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

trainer.remove_callback(NotebookProgressCallback)

print(trainer.evaluate())

{'eval_loss': 1.2395516633987427, 'eval_runtime': 9.2559, 'eval_samples_per_second': 110.2, 'eval_steps_per_second': 27.55, 'epoch': 1.0}


In [ ]:
import os

LORA_DIR = "/content/bert_squad_finetuned"

print(os.listdir(LORA_DIR))

['tokenizer.json', 'checkpoint-5041', 'model.safetensors', 'tokenizer_config.json', 'training_args.bin', 'config.json']


### Final validation EM(Exact Match) / F1_score

In [ ]:
pred_output = trainer.predict(validation_features)
start_logits, end_logits = pred_output.predictions

final_metrics = compute_squad_metrics(
    start_logits=start_logits,
    end_logits=end_logits,
    features=validation_features,
    examples=raw_datasets["validation"],
)

print("\nFinal SQuAD metrics:")
print(final_metrics)

  0%|          | 0/1000 [00:00<?, ?it/s]


Final SQuAD metrics:
{'exact_match': 78.0, 'f1': 83.65370329343855}


### Inference test

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

# Load saved fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
model = AutoModelForQuestionAnswering.from_pretrained(OUTPUT_DIR)
model.eval()

def answer_question(question, context, max_length=384):
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = model(**inputs)

    start_idx = torch.argmax(outputs.start_logits).item()
    end_idx = torch.argmax(outputs.end_logits).item()

    if end_idx < start_idx:
        end_idx = start_idx

    answer_ids = inputs["input_ids"][0][start_idx:end_idx + 1]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "start_token": start_idx,
        "end_token": end_idx
    }

# Example
context = """
BERT stands for Bidirectional Encoder Representations from Transformers.
It is a transformer-based language model developed by Google and
introduced in 2018. BERT understands the context of words by looking
at both the left and right sides of a word.

Machine Learning (ML) is a subset of Artificial Intelligence (AI).
It enables computers to learn patterns from data and make predictions
without being explicitly programmed for every task.

Deep Learning (DL) is a subset of Machine Learning that uses neural
networks with multiple layers to learn complex patterns from data.
Deep Learning is widely used in image recognition, speech recognition,
and natural language processing.
"""

questions = [
    "What is BERT?",
    "Who developed BERT?",
    "When was BERT introduced?",
    "What is ML?",
    "What is DL?",
    "What does BERT stand for?",
    "How does BERT understand words?",
    "What is the relationship between ML and AI?",
    "What does deep learning use?",
    "Where is deep learning used?"
]

for question in questions:
    result = answer_question(question, context)

    print("Question:", result["question"])
    print("Answer:", result["answer"])
    print("Start token:", result["start_token"])
    print("End token:", result["end_token"])
    print("-" * 50)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Question: What is BERT?
Answer: bidirectional encoder representations from transformers
Start token: 9
End token: 18
--------------------------------------------------
Question: Who developed BERT?
Answer: google
Start token: 31
End token: 31
--------------------------------------------------
Question: When was BERT introduced?
Answer: 2018
Start token: 36
End token: 36
--------------------------------------------------
Question: What is ML?
Answer: machine learning
Start token: 56
End token: 57
--------------------------------------------------
Question: What is DL?
Answer: deep learning
Start token: 90
End token: 91
--------------------------------------------------
Question: What does BERT stand for?
Answer: bidirectional encoder representations from transformers
Start token: 11
End token: 20
--------------------------------------------------
Question: How does BERT understand words?
Answer: looking
Start token: 46
End token: 46
--------------------------------------------------
Que

## If we apply:
LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that freezes the original pre-trained model weights and trains only small additional matrices. It reduces the number of trainable parameters, optimizer memory usage, and often training time. In my BERT question-answering project, LoRA reduced the training time per epoch from approximately 9 minutes to 5–6 minutes, with only 0.27% of the model parameters trainable.